# 03 — Dose Projection from IC50

Projects efficacious doses for preclinical animal studies based on:
- In vitro potency (IC50)
- Clearance (from PK studies)
- Plasma protein binding (fu)
- Oral bioavailability (F)

**Key equation:**
```
Dose = (IC50 × CL × τ) / (fu × F)
```

In [ ]:
import sys
sys.path.insert(0, '..')

from doseprojection.dose_projection import (
    efficacious_dose_mg_kg, project_animal_dose,
    steady_state_css, unbound_concentration
)
from doseprojection.io import load_invitro_data, load_pk_data
import pandas as pd

## Single Compound Example

CPD-001: IC50 = 15 nM, MW = 450, rat CL = 25 mL/min/kg, fu = 0.08, F = 45%

In [ ]:
result = project_animal_dose(
    ic50_nm=15,
    mw=450,
    cl_animal_mL_min_kg=25,
    fu_animal=0.08,
    tau_h=24,
    f_animal=0.45,
    coverage_multiple=1.0,  # 1x IC50 coverage at Css,avg
    route="po"
)

print("=== CPD-001 Dose Projection (Rat, PO, QD) ===")
print(f"Projected dose (1x IC50): {result['dose_mg_kg']:.1f} mg/kg")
print(f"Target Cu: {result['target_cu_ng_mL']:.2f} ng/mL")
print(f"Predicted Css,total: {result['predicted_css_total_ng_mL']:.0f} ng/mL")

## Coverage Multiple Analysis

Explore dose requirements at different IC50 coverage multiples.

In [ ]:
multiples = [0.5, 1, 3, 5, 10]
rows = []
for m in multiples:
    dose = efficacious_dose_mg_kg(
        ic50_nm=15, mw=450, cl_mL_min_kg=25,
        fu=0.08, tau_h=24, f=0.45, coverage_multiple=m
    )
    css = steady_state_css(dose, 25, 24, f=0.45, route="po")
    cu = unbound_concentration(css, 0.08)
    rows.append({
        'Coverage': f'{m}x IC50',
        'Dose (mg/kg)': round(dose, 1),
        'Css,total (ng/mL)': round(css, 0),
        'Cu,ss (ng/mL)': round(cu, 2),
    })

pd.DataFrame(rows)

## Batch Projection from Data Files

Load all compounds and project doses using their measured PK parameters.

In [ ]:
invitro = load_invitro_data('../examples/example_invitro_data.csv')
pk = load_pk_data('../examples/example_pk_data.csv')

# Get rat IV CL and rat PO F for each compound
rat_iv = pk[(pk['species'] == 'rat') & (pk['route'] == 'IV')][['compound_id', 'CL_mL_min_kg']]
rat_po = pk[(pk['species'] == 'rat') & (pk['route'] == 'PO')][['compound_id', 'F_pct']]

df = invitro.merge(rat_iv, on='compound_id', how='inner')
df = df.merge(rat_po, on='compound_id', how='inner')

# Project doses
results = []
for _, row in df.iterrows():
    dose = efficacious_dose_mg_kg(
        ic50_nm=row['IC50_nM'],
        mw=row['MW'],
        cl_mL_min_kg=row['CL_mL_min_kg'],
        fu=row['fu_plasma_rat'],
        tau_h=24,
        f=row['F_pct'] / 100,
        route="po"
    )
    results.append({
        'Compound': row['compound_id'],
        'IC50 (nM)': row['IC50_nM'],
        'fu (rat)': row['fu_plasma_rat'],
        'CL (mL/min/kg)': row['CL_mL_min_kg'],
        'F (%)': row['F_pct'],
        'Projected Dose (mg/kg)': round(dose, 1),
    })

pd.DataFrame(results)